# NEURALSHIELD AI // PHASE 2: ADVANCED EXPLORATORY DATA ANALYSIS
## Intelligent Cyber Threat Detection & Network Intelligence Platform

**Primary Dataset:** CIC-IDS2017 (Canadian Institute for Cybersecurity)  
**Document:** `notebooks/02_advanced_exploratory_data_analysis.ipynb`  
**Phase:** Phase 2 — Advanced Exploratory Data Analysis & Empirical Threat Profiling

---
### Advanced EDA Objectives
1. Profile dataset dimensions, data types, and 5 semantic feature groups.
2. Quantify extreme class imbalance between benign flows and minority attack categories.
3. Conduct empirical statistical comparisons between benign and malicious traffic across durations, volume, and throughput rates.
4. Contrast TCP handshake telemetry (SYN, ACK, RST flags) to detect reconnaissance and flooding patterns.
5. Identify severe multicollinearity and map out leakage safeguards prior to Phase 3 ML model training.

*Provenance Note: In accordance with platform design standards, if full raw benchmark CSVs are not present in `data/raw/cic_ids2017/`, this notebook seamlessly profiles the verified demonstration dataset in `data/sample/`. The active mode is explicitly indicated.*

In [ ]:
import sys
from pathlib import Path

# Path configuration
PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from neuralshield.data.discovery import inspect_dataset_inventory
from neuralshield.data.loader import DatasetLoader, downcast_numeric_dtypes, sample_dataframe
from neuralshield.data.schema import compare_schemas
from neuralshield.data.cleaning import (
    normalize_column_names,
    identify_and_drop_redundant_columns,
    sanitize_infinite_and_nan,
    handle_duplicates,
    normalize_labels,
)
from neuralshield.data.validation import audit_dataframe_quality
from neuralshield.analysis.statistics import compute_descriptive_statistics
from neuralshield.analysis.distributions import analyze_class_balance
from neuralshield.analysis.correlations import compute_correlation_matrix, find_high_correlation_pairs
from neuralshield.analysis.comparisons import compare_benign_vs_attack
from neuralshield.features.engineering import (
    create_network_domain_features,
    remove_target_leakage_features,
    get_feature_catalog,
)

# Dark command-center visual style
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = 'Segoe UI, Inter, sans-serif'
print("Phase 2 environment initialized successfully.")

### 1. Ingestion & Schema Profiling

In [ ]:
raw_dir = PROJECT_ROOT / "data" / "raw" / "cic_ids2017"
sample_file = PROJECT_ROOT / "data" / "sample" / "cic_ids2017_sample.parquet"
sample_csv = PROJECT_ROOT / "data" / "sample" / "cic_ids2017_sample.csv"

inv = inspect_dataset_inventory(raw_dir)
print(f"Directory Status: {inv['status']}")
print(f"Benchmark Files Present: {inv['expected_present_count']} / {inv['expected_benchmark_files']}")

loader = DatasetLoader()
if inv["total_files_found"] > 0:
    first_csv = inv["expected_inventory"][0]["path"]
    print(f"Loading raw file: {first_csv}")
    df_raw = loader.load_single_file(first_csv, nrows=50000, optimize_memory=True)
    data_mode = "RAW_BENCHMARK"
elif sample_file.is_file():
    print(f"Loading synthetic sample: {sample_file}")
    df_raw = pd.read_parquet(sample_file)
    data_mode = "SYNTHETIC_SAMPLE"
else:
    print(f"Loading CSV sample: {sample_csv}")
    df_raw = pd.read_csv(sample_csv)
    data_mode = "SYNTHETIC_SAMPLE"

print(f"Active Dataset Mode: {data_mode}")
print(f"Dimensions: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

### 2. Semantic Feature Inventory & Catalog
Organizing network features into the 5 core analytical groups: Volume, Timing, TCP Behavior, Rates, and Packet Characteristics.

In [ ]:
df_clean, _ = normalize_column_names(df_raw)
catalog = get_feature_catalog(df_clean)
for group_id, meta in catalog.items():
    print(f"[{meta['group_name'].upper()}] - {meta['present_features_count']}/{meta['expected_features_count']} Features Mapped")
    print(f"  Description: {meta['description']}")
    print(f"  Mapped: {meta['present_features'][:4]}...
")

### 3. Extended Descriptive Statistics
Computing mean, standard deviation, median, IQR, variance, skewness, and kurtosis to identify heavy-tailed distributions.

In [ ]:
desc_stats = compute_descriptive_statistics(df_clean)
display(desc_stats.head(10))

### 4. Class Distribution & Imbalance Audit
Evaluating majority versus minority attack distributions to inform Phase 3 resampling and evaluation metric selection.

In [ ]:
target_col = "label"
df_clean, label_audit = normalize_labels(df_clean, target_col=target_col)
balance = analyze_class_balance(df_clean, target_col=target_col)

print(f"Total Flows: {balance['total_samples']:,}")
print(f"Majority Class: {balance['majority_class']} ({balance['class_percentages'].get(balance['majority_class'], 0)}%)")
print(f"Minority Class: {balance['minority_class']} ({balance['class_percentages'].get(balance['minority_class'], 0)}%)")
print(f"Imbalance Ratio: {balance['imbalance_ratio']}:1")

# Class distribution visualization
plt.figure(figsize=(9, 4))
cats = list(balance["class_counts"].keys())
counts = list(balance["class_counts"].values())
bar_colors = ['#16C784' if 'benign' in str(c).lower() else '#EF4444' for c in cats]

plt.bar(cats, counts, color=bar_colors, edgecolor='#1D2824')
plt.title("Traffic Category Distribution", fontsize=14, color='#E8F0EB', pad=12)
plt.ylabel("Observed Flows", color='#E8F0EB')
plt.xticks(rotation=20, ha='right', color='#8B9A91')
plt.grid(axis='y', color='#1D2824', linestyle='--')
plt.tight_layout()
plt.show()

### 5. Empirical Attack Behavior Contrast
Contrasting behavioral signatures between normal baseline flows and attack flows.

In [ ]:
comparison = compare_benign_vs_attack(df_clean, target_col=target_col)
if "warning" in comparison:
    print(comparison["warning"])
else:
    print(f"Benign Flows Analyzed: {comparison['benign_sample_count']:,}")
    print(f"Attack Flows Analyzed: {comparison['attack_sample_count']:,}
")

    comp_df = pd.DataFrame(comparison["feature_comparisons"]).T
    display(comp_df[["benign_median", "attack_median", "attack_to_benign_median_ratio"]])

    print("
Empirical Findings:")
    for finding in comparison["empirical_findings"]:
        print(f"  * {finding}")

### 6. TCP Handshake Flag Signatures
Analyzing flag activation percentages across normal sessions and reconnaissance/flooding attacks.

In [ ]:
if "tcp_flag_comparisons" in comparison and comparison["tcp_flag_comparisons"]:
    flag_df = pd.DataFrame(comparison["tcp_flag_comparisons"]).T
    display(flag_df)

    plt.figure(figsize=(10, 4))
    flags = list(flag_df.index)
    x = np.arange(len(flags))
    width = 0.35

    plt.bar(x - width/2, flag_df["benign_percentage_active"], width, label='Benign', color='#16C784')
    plt.bar(x + width/2, flag_df["attack_percentage_active"], width, label='Attack', color='#EF4444')
    plt.xticks(x, [f.replace('_', ' ').title() for f in flags], color='#8B9A91')
    plt.ylabel("% Flows with Flag Active", color='#E8F0EB')
    plt.title("TCP Flag Telemetry: Benign vs Attack Traffic", color='#E8F0EB', fontsize=14)
    plt.legend()
    plt.grid(axis='y', color='#1D2824', linestyle='--')
    plt.tight_layout()
    plt.show()

### 7. Correlation Analysis & Multicollinearity Screening
Identifying redundant features with $|r| \ge 0.90$ to avoid variance inflation in tree-based and linear models.

In [ ]:
corr = compute_correlation_matrix(df_clean, max_features=14)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Network Flow Feature Correlation Heatmap", color='#E8F0EB', fontsize=14, pad=12)
plt.xticks(rotation=45, ha='right', color='#8B9A91')
plt.yticks(color='#8B9A91')
plt.tight_layout()
plt.show()

collinear = find_high_correlation_pairs(corr, threshold=0.90)
print(f"Detected {len(collinear)} Highly Collinear Feature Pairs (|r| >= 0.90):")
for p in collinear[:6]:
    print(f"  - {p['feature_1']} <--> {p['feature_2']}: r = {p['correlation']}")

### 8. Domain Feature Engineering Execution

In [ ]:
df_feat = create_network_domain_features(df_clean)
new_features = [c for c in df_feat.columns if c not in df_clean.columns]
print(f"Generated {len(new_features)} Domain Features: {new_features}")
display(df_feat[new_features].head())

### 9. Target Leakage Isolation
Verifying that network identifiers (`Flow ID`, `Source IP`, `Destination IP`, `Timestamp`) are stripped prior to model preparation.

In [ ]:
df_ml, stripped = remove_target_leakage_features(df_feat)
print(f"Stripped Target-Leakage Features: {stripped}")
print(f"ML Feature Space Count: {len(df_ml.columns)}")

## Phase 2 Analytical Conclusions & Phase 3 ML Strategy

### Analytical Findings
1. **Flow Duration Asymmetry:** Automated attacks exhibit heavily truncated flow durations compared to sustained benign sessions.
2. **SYN Density Divergence:** PortScan and SYN floods show near-zero backward packet responses accompanied by elevated SYN flag counts.
3. **Heavy Tailed Telemetry:** Duration and volume spans 5+ orders of magnitude; log-scaling (`log_flow_duration`) is crucial for model convergence.
4. **Multicollinearity:** Strong collinearity exists between forward bytes, packet lengths, and header counts. Redundant features should be pruned before linear modeling.

### Phase 3 ML Architecture Directives
- **Splitting Strategy:** Avoid simple random train/test splits due to temporal correlation. Use day-based splits or stratified temporal partitions.
- **Resampling:** Prohibit global SMOTE. Apply resampling strictly to training folds.
- **Target Metrics:** Optimize for Precision-Recall AUC (PR-AUC) and False Positive Rate rather than raw accuracy.